# Layer 6: Infrastructure Design Translation
Translates measured throughput into engineering specifications.

In [ ]:
import os, json, numpy as np, pandas as pd

DATA_DIR    = os.environ.get("DATA_DIR",    "/data/parquet")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/data/reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

from mda.loader import load_trades, load_orderbook, load_orderbook_updates
from mda.layer1.rates import layer1_summary
from mda.layer2.volleys import detect_volleys, volley_size_distribution
from mda.layer3.execution_rate import compute_execution_rate, execution_rate_stats
from mda.layer3.microbursts import detect_microbursts, microburst_stats
from mda.layer3.sequencing import compute_out_of_order
from mda.layer3.quantisation import resolution_report
from mda.layer4.update_rates import compute_update_rate_by_depth
from mda.layer6.design_specs import compute_specs, write_design_spec
print("imports OK")

In [ ]:
df = load_trades(DATA_DIR)
print(f"Loaded {len(df):,} trades")

## Build layer summary dicts

In [ ]:
l1 = layer1_summary(df)
print("L1 done:", list(l1.keys()))

In [ ]:
_, volley_stats = detect_volleys(df)
l2 = {"volley_stats": volley_stats}
print(f"L2 done: {len(volley_stats):,} volleys")

In [ ]:
exec_rate  = compute_execution_rate(df)
exec_stats = execution_rate_stats(exec_rate)
bursts     = detect_microbursts(df)
mb_s       = microburst_stats(bursts)
oo_summary, _ = compute_out_of_order(df)
res_report = resolution_report(df)
l3 = {
    "exec_rate_stats": exec_stats,
    "microburst_stats": mb_s,
    "oo_summary": oo_summary,
    "resolution_report": res_report,
}
print("L3 done:", {k: len(v) for k,v in l3.items()})

In [ ]:
import gc, pandas as pd

# Per-exchange minimal column load — same pattern as L4/L5.
# Full OB is 24 GB for Binance; [exchange, level, received_at] = ~1 GB.
_ob_exchanges = df["exchange"].unique().tolist()
_ob_parts = []
for _exch in _ob_exchanges:
    _ob_exch = load_orderbook(DATA_DIR, exchanges=[_exch],
                              columns=["exchange", "level", "received_at"],
                              add_ts_cols=False)
    _ob_exch["receive_ts_dt"] = pd.to_datetime(
        _ob_exch["received_at"], utc=True, format="mixed")
    _ob_parts.append(compute_update_rate_by_depth(_ob_exch))
    del _ob_exch
    gc.collect()

ob_update_rate = pd.concat(_ob_parts, ignore_index=True)
del _ob_parts
gc.collect()
l4 = {"ob_update_rate": ob_update_rate}
print(f"L4 done: {len(ob_update_rate):,} update rate rows")

In [ ]:
specs = compute_specs(l1, l2, l3, l4)
print(json.dumps(specs, indent=2))

In [ ]:
spec_path = os.path.join(REPORTS_DIR, "design_spec.md")
write_design_spec(specs, spec_path)

In [ ]:
from IPython.display import Markdown, display
with open(spec_path) as f:
    display(Markdown(f.read()))

In [ ]:
pd.DataFrame(list(specs.items()), columns=["Parameter", "Value"])